# MM-Fit: RandomForest vs 1D-CNN (Klassifikation)

Dieses Notebook erstellt Sliding-Window-Segmente aus dem MM-Fit Datensatz und vergleicht eine RandomForest-Baseline
mit einem 1D-CNN.


## 1) Setup und Reproduzierbarkeit


In [1]:
import os
import math
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, ParameterSampler

import tensorflow as tf
from tensorflow.keras import layers, models

# Reproduzierbarkeit
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Pfade und Konfiguration
ROOT = "mm-fit"
TARGETS = ["pushups", "squats", "situps"]
label_map = {"pushups": 0, "squats": 1, "situps": 2}
inv_label_map = {v: k for k, v in label_map.items()}

# Daten-/Feature-Config (Finales Setup)
# SENSOR_MODE:
# - "manual": nutzt SENSORS_MANUAL
# - "auto_common": nimmt Sensoren, die in allen Sessions existieren (max SENSOR_MAX_COUNT)
SENSOR_MODE = "manual"
SENSORS_MANUAL = ["sw_r"]
SENSOR_MAX_COUNT = 3
SENSOR_PRIORITY = ["sw_r", "sw_l", "thigh_r", "thigh_l", "waist", "head"]
SENSORS = None  # wird in der Ladezelle aufgeloest

# Bestes Windowing aus Abschnitt 13
WIN = 192
STEP = 64

# Split-Config
TEST_SIZE = 0.20
VAL_SIZE_OF_REST = 0.20

# Modell-/Trainings-Defaults
MODEL_VARIANT = "resnet"  # "baseline" oder "resnet"
BASE_FILTERS = 64
DROPOUT_HEAD = 0.40
DENSE_UNITS = 128
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
USE_FOCAL_LOSS = False
FOCAL_GAMMA = 2.0

print("OK - Setup loaded")


OK - Setup loaded


/Users/kacharino/myProjects/Bachelor/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


**Datensatz**
- Root: `mm-fit/` mit Sessions `w00`–`w20`
- Pro Session: `*_sw_r_acc.npy`, `*_sw_r_gyr.npy`, `*_labels.csv`
- Labels: `start, end, reps, exercise` (nur pushups/squats/situps)


## 2) Sessions und dynamischer Split (train/val/test, gruppiert nach Session)


Train/Val/Test wird zufällig und reproduzierbar über Sessions gesplittet.
Alle Windows einer Session bleiben im selben Split (kein Leakage).
Zusätzlich sind jetzt Multi-Sensor-Feature-Fusion, größere CNN-Varianten und robuste Mehrfach-Evaluation enthalten.


In [2]:
def list_sessions(root=ROOT):
    return sorted([
        d for d in os.listdir(root)
        if d.startswith("w") and os.path.isdir(os.path.join(root, d))
    ])

sessions = list_sessions(ROOT)

# 1) Test split auf Session-Ebene
outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
)
idx_train_val, idx_test = next(outer_split.split(sessions, groups=sessions))

train_val_sessions = [sessions[i] for i in idx_train_val]
test_sessions = [sessions[i] for i in idx_test]

# 2) Val split innerhalb des verbleibenden Train-Val-Teils
inner_split = GroupShuffleSplit(
    n_splits=1,
    test_size=VAL_SIZE_OF_REST,
    random_state=RANDOM_SEED,
)
idx_train, idx_val = next(inner_split.split(train_val_sessions, groups=train_val_sessions))

train_sessions = [train_val_sessions[i] for i in idx_train]
val_sessions = [train_val_sessions[i] for i in idx_val]

print("Anzahl Sessions:", len(sessions))
print("Train sessions (n=%d):" % len(train_sessions), train_sessions)
print("Val sessions   (n=%d):" % len(val_sessions), val_sessions)
print("Test sessions  (n=%d):" % len(test_sessions), test_sessions)


Anzahl Sessions: 21
Train sessions (n=12): ['w04', 'w05', 'w06', 'w09', 'w10', 'w11', 'w12', 'w13', 'w14', 'w16', 'w18', 'w20']
Val sessions   (n=4): ['w02', 'w03', 'w07', 'w19']
Test sessions  (n=5): ['w00', 'w01', 'w08', 'w15', 'w17']


## 3) Laden einer Session (ACC + GYR, x/y/z)


In [3]:
def list_available_sensors(root=ROOT):
    sessions = list_sessions(root)
    if not sessions:
        return []

    first_session = sessions[0]
    base = os.path.join(root, first_session)
    sensors = set()

    for fname in os.listdir(base):
        if not fname.startswith(f"{first_session}_"):
            continue
        if fname.endswith("_acc.npy"):
            sensors.add(fname.replace(f"{first_session}_", "").replace("_acc.npy", ""))

    return sorted(sensors)


def sensors_in_session(wdir, root=ROOT):
    base = os.path.join(root, wdir)
    sensors = set()
    for fname in os.listdir(base):
        if not fname.startswith(f"{wdir}_"):
            continue
        if fname.endswith("_acc.npy"):
            s = fname.replace(f"{wdir}_", "").replace("_acc.npy", "")
            gyr_path = os.path.join(base, f"{wdir}_{s}_gyr.npy")
            if os.path.exists(gyr_path):
                sensors.add(s)
    return sensors


def get_common_sensors(root=ROOT):
    sessions = list_sessions(root)
    if not sessions:
        return []

    common = None
    for wdir in sessions:
        sset = sensors_in_session(wdir, root=root)
        common = sset if common is None else (common & sset)

    return sorted(common) if common else []


def rank_sensors(sensor_list, priority=None):
    priority = priority or []
    rank = {s: i for i, s in enumerate(priority)}
    return sorted(sensor_list, key=lambda s: (rank.get(s, 10_000), s))


def resolve_active_sensors(
    mode=SENSOR_MODE,
    manual=SENSORS_MANUAL,
    max_count=SENSOR_MAX_COUNT,
    priority=SENSOR_PRIORITY,
):
    if mode == "manual":
        chosen = list(manual)
    elif mode == "auto_common":
        common = get_common_sensors(ROOT)
        chosen = rank_sensors(common, priority=priority)
        if max_count is not None:
            chosen = chosen[:max_count]
    else:
        raise ValueError(f"Unbekannter SENSOR_MODE: {mode}")

    if not chosen:
        chosen = ["sw_r"]

    return chosen


def load_one_session(wdir, sensors=None):
    sensors = sensors or SENSORS
    base = os.path.join(ROOT, wdir)

    feature_streams = []
    lengths = []

    for sensor in sensors:
        acc_path = os.path.join(base, f"{wdir}_{sensor}_acc.npy")
        gyr_path = os.path.join(base, f"{wdir}_{sensor}_gyr.npy")

        if not (os.path.exists(acc_path) and os.path.exists(gyr_path)):
            raise FileNotFoundError(f"Sensor '{sensor}' fehlt in Session '{wdir}'.")

        acc = np.load(acc_path)
        gyr = np.load(gyr_path)

        # x, y, z (Spalten 1 bis 3)
        acc_xyz = acc[:, 1:4].astype(np.float32)
        gyr_xyz = gyr[:, 1:4].astype(np.float32)

        # Manche Sessions haben minimale Längendifferenzen zwischen acc/gyr
        n = min(len(acc_xyz), len(gyr_xyz))
        acc_xyz = acc_xyz[:n]
        gyr_xyz = gyr_xyz[:n]

        feature_streams.append(np.hstack([acc_xyz, gyr_xyz]))
        lengths.append(n)

    # Falls mehrere Sensoren leicht unterschiedliche Längen haben: global trimmen
    n_global = min(lengths)
    feature_streams = [fs[:n_global] for fs in feature_streams]

    # Pro Sensor 6 Kanäle, bei mehreren Sensoren Kanäle konkatenieren
    features = np.hstack(feature_streams).astype(np.float32)

    labels = pd.read_csv(
        os.path.join(base, f"{wdir}_labels.csv"),
        header=None,
        names=["start", "end", "reps", "exercise"],
    )
    labels = labels[labels["exercise"].isin(TARGETS)].reset_index(drop=True)

    # Labelgrenzen auf verfügbare Signal-Länge begrenzen
    labels["start"] = labels["start"].clip(lower=0, upper=n_global)
    labels["end"] = labels["end"].clip(lower=0, upper=n_global)
    labels = labels[labels["end"] > labels["start"]].reset_index(drop=True)

    return features, labels


common = get_common_sensors(ROOT)
SENSORS = resolve_active_sensors()
print("Verfügbare Sensoren (Beispielsession):", list_available_sensors())
print("Gemeinsame Sensoren über alle Sessions:", common)
print("SENSOR_MODE:", SENSOR_MODE)
print("Aktive Sensoren:", SENSORS)


Verfügbare Sensoren (Beispielsession): ['eb_l', 'sp_r', 'sw_l', 'sw_r']
Gemeinsame Sensoren über alle Sessions: ['eb_l', 'sp_r', 'sw_l', 'sw_r']
SENSOR_MODE: manual
Aktive Sensoren: ['sw_r']


## 4) Windowing (win=128, step=64)


In [4]:
def make_windows(features, labels_df, win=WIN, step=STEP):
    X, y = [], []
    for _, row in labels_df.iterrows():
        start, end = int(row["start"]), int(row["end"])
        lab = label_map[row["exercise"]]

        for i in range(start, end - win + 1, step):
            X.append(features[i:i+win])
            y.append(lab)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


## 5) Dataset bauen

In [5]:
def build_from_session(wdir, sensors=None):
    features, labels = load_one_session(wdir, sensors=sensors)
    X, y = make_windows(features, labels)
    return X, y


def build_dataset(session_list, split_name="split", sensors=None, verbose=True):
    X_all, y_all = [], []
    for wdir in session_list:
        X, y = build_from_session(wdir, sensors=sensors)
        if len(X) == 0:
            if verbose:
                print("Skip (no windows):", wdir)
            continue
        X_all.append(X)
        y_all.append(y)
        if verbose:
            print(f"[{split_name}]", wdir, "->", X.shape, np.unique(y, return_counts=True))

    if not X_all:
        raise ValueError(f"No windows found for split '{split_name}'.")

    return np.concatenate(X_all), np.concatenate(y_all)

X_train, y_train = build_dataset(train_sessions, split_name="train", sensors=SENSORS)
X_val, y_val = build_dataset(val_sessions, split_name="val", sensors=SENSORS)
X_test, y_test = build_dataset(test_sessions, split_name="test", sensors=SENSORS)

print("\nFINAL:")
print("X_train:", X_train.shape, "y_train:", y_train.shape, np.unique(y_train, return_counts=True))
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape,   np.unique(y_val, return_counts=True))
print("X_test :", X_test.shape,  "y_test :", y_test.shape,  np.unique(y_test, return_counts=True))
print("Feature-Kanäle:", X_train.shape[-1])


[train] w04 -> (77, 192, 6) (array([0, 1, 2]), array([14, 28, 35]))
[train] w05 -> (48, 192, 6) (array([0, 1, 2]), array([13, 19, 16]))
[train] w06 -> (56, 192, 6) (array([0, 1, 2]), array([17, 19, 20]))
[train] w09 -> (77, 192, 6) (array([0, 1, 2]), array([18, 25, 34]))
[train] w10 -> (58, 192, 6) (array([0, 1, 2]), array([16, 20, 22]))
[train] w11 -> (83, 192, 6) (array([0, 1, 2]), array([18, 26, 39]))
[train] w12 -> (50, 192, 6) (array([0, 1, 2]), array([10, 18, 22]))
[train] w13 -> (53, 192, 6) (array([0, 1, 2]), array([10, 17, 26]))
[train] w14 -> (49, 192, 6) (array([0, 1, 2]), array([14, 16, 19]))
[train] w16 -> (77, 192, 6) (array([0, 1, 2]), array([28, 20, 29]))
[train] w18 -> (83, 192, 6) (array([0, 1, 2]), array([17, 27, 39]))
[train] w20 -> (83, 192, 6) (array([0, 1, 2]), array([22, 31, 30]))
[val] w02 -> (71, 192, 6) (array([0, 1, 2]), array([14, 24, 33]))
[val] w03 -> (50, 192, 6) (array([0, 1, 2]), array([15, 15, 20]))
[val] w07 -> (81, 192, 6) (array([0, 1, 2]), array([

## 6) Normalisierung (train-basiert)


In [6]:
mu = X_train.mean(axis=(0, 1), keepdims=True)
sigma = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train_n = (X_train - mu) / sigma
X_val_n   = (X_val   - mu) / sigma
X_test_n  = (X_test  - mu) / sigma

print("Normalized shapes:", X_train_n.shape, X_val_n.shape, X_test_n.shape)
print("Train mean ~", X_train_n.mean(), "Train std ~", X_train_n.std())


Normalized shapes: (794, 192, 6) (275, 192, 6) (300, 192, 6)
Train mean ~ 0.027537148 Train std ~ 0.9996554


In [7]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print("class_weights:", class_weights)


class_weights: {np.int64(0): np.float64(1.3434856175972927), np.int64(1): np.float64(0.9949874686716792), np.int64(2): np.float64(0.7995971802618328)}


## 7) RandomForest 


In [8]:
X_train_rf = X_train_n.reshape(X_train_n.shape[0], -1)
X_test_rf  = X_test_n.reshape(X_test_n.shape[0], -1)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

rf.fit(X_train_rf, y_train)
y_pred_rf = rf.predict(X_test_rf)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1_macro = f1_score(y_test, y_pred_rf, average="macro")

print("Random Forest Accuracy:", rf_acc)
print("Random Forest Macro-F1:", rf_f1_macro)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=[inv_label_map[i] for i in sorted(inv_label_map)],
    zero_division=0,
))


Random Forest Accuracy: 0.5033333333333333
Random Forest Macro-F1: 0.41133758336458354

Confusion Matrix:
[[ 5 35 26]
 [10 52 42]
 [11 25 94]]

Classification Report:
              precision    recall  f1-score   support

     pushups       0.19      0.08      0.11        66
      squats       0.46      0.50      0.48       104
      situps       0.58      0.72      0.64       130

    accuracy                           0.50       300
   macro avg       0.41      0.43      0.41       300
weighted avg       0.45      0.50      0.47       300



## 8) 1D-CNN (Baseline + Residual/Stacked Variante)


In [9]:
def make_sparse_focal_loss(gamma=2.0):
    def focal_loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_onehot = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        ce = -tf.reduce_sum(y_true_onehot * tf.math.log(y_pred), axis=-1)
        p_t = tf.reduce_sum(y_true_onehot * y_pred, axis=-1)
        modulating = tf.pow(1.0 - p_t, gamma)
        return tf.reduce_mean(modulating * ce)

    return focal_loss


def residual_block(x, filters, kernel_size=3, stride=1, weight_decay=1e-5):
    shortcut = x

    y = layers.Conv1D(
        filters,
        kernel_size,
        strides=stride,
        padding="same",
        kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
    )(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)

    y = layers.Conv1D(
        filters,
        kernel_size,
        padding="same",
        kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
    )(y)
    y = layers.BatchNormalization()(y)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(
            filters,
            1,
            strides=stride,
            padding="same",
            kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    out = layers.Add()([y, shortcut])
    out = layers.ReLU()(out)
    return out


def build_cnn(
    input_shape,
    num_classes,
    model_variant=MODEL_VARIANT,
    base_filters=BASE_FILTERS,
    dropout_head=DROPOUT_HEAD,
    dense_units=DENSE_UNITS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    use_focal_loss=USE_FOCAL_LOSS,
    focal_gamma=FOCAL_GAMMA,
):
    inputs = layers.Input(shape=input_shape)

    if model_variant == "baseline":
        x = layers.Conv1D(base_filters, 7, padding="same")(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling1D(2)(x)

        x = layers.Conv1D(base_filters * 2, 5, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling1D(2)(x)

        x = layers.Conv1D(base_filters * 4, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    else:
        # Stacked Residual 1D-CNN
        x = layers.Conv1D(
            base_filters,
            7,
            padding="same",
            kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
        )(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        x = residual_block(x, base_filters, kernel_size=7, stride=1, weight_decay=weight_decay)
        x = residual_block(x, base_filters * 2, kernel_size=5, stride=2, weight_decay=weight_decay)
        x = residual_block(x, base_filters * 2, kernel_size=5, stride=1, weight_decay=weight_decay)
        x = residual_block(x, base_filters * 4, kernel_size=3, stride=2, weight_decay=weight_decay)
        x = residual_block(x, base_filters * 4, kernel_size=3, stride=1, weight_decay=weight_decay)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout_head)(x)
    x = layers.Dense(
        dense_units,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
    )(x)
    x = layers.Dropout(dropout_head * 0.8)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    loss_fn = make_sparse_focal_loss(focal_gamma) if use_focal_loss else "sparse_categorical_crossentropy"

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=loss_fn,
        metrics=["accuracy"],
    )
    return model


cnn = build_cnn(input_shape=(WIN, X_train_n.shape[-1]), num_classes=len(label_map))
cnn.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 192, 6)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 192, 64)   │      2,752 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 192, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 192, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 192, 64)   │     28,736 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 192, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 192, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 192, 64)   │     28,736 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 192, 64)   │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 192, 64)   │          0 │ batch_normalizat… │
│                     │                   │            │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 192, 64)   │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 96, 128)   │     41,088 │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 96, 128)   │        512 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 96, 128)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 96, 128)   │     82,048 │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 96, 128)   │      8,320 │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 96, 128)   │        512 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 96, 128)   │        512 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 96, 128)   │          0 │ batch_normalizat… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 96, 128)   │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,119,683 (4.27 MB)

 Trainable params: 1,115,459 (4.26 MB)

 Non-trainable params: 4,224 (16.50 KB)

**Hinweis:** Train/Val/Test ist session-basiert (kein Window-Leakage).
CNN nutzt jetzt val-loss-basiertes EarlyStopping/Checkpointing, optionale Focal Loss und eine größere Residual-Architektur.


In [10]:
# Data augmentation: jitter + scaling
# (applied on-the-fly to training windows)
def augment(x, y):
    noise = tf.random.normal(tf.shape(x), mean=0.0, stddev=0.02)
    scale = tf.random.uniform([tf.shape(x)[0], 1, 1], 0.9, 1.1)
    x = x * scale + noise
    return x, y

batch_size = 32

X_tr, y_tr = X_train_n, y_train
X_val_split, y_val_split = X_val_n, y_val

train_ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
train_ds = train_ds.shuffle(min(8192, len(X_tr)), seed=RANDOM_SEED).batch(batch_size).map(augment).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val_split, y_val_split))
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

checkpoint_path = "best_cnn_single_split.keras"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=0,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
    ),
]

class_weights_for_fit = None if USE_FOCAL_LOSS else class_weights

history = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    class_weight=class_weights_for_fit,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.4370 - loss: 1.4175 - val_accuracy: 0.3927 - val_loss: 1.1082 - learning_rate: 3.0000e-04
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.5479 - loss: 1.0918 - val_accuracy: 0.4764 - val_loss: 1.2621 - learning_rate: 3.0000e-04
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.5655 - loss: 0.9902 - val_accuracy: 0.5636 - val_loss: 0.9821 - learning_rate: 3.0000e-04
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.6171 - loss: 0.9017 - val_accuracy: 0.5273 - val_loss: 1.0171 - learning_rate: 3.0000e-04
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.6121 - loss: 0.8776 - val_accuracy: 0.3418 - val_loss: 1.1584 - learning_rate: 3.0000e-04
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.6234 - loss: 0.8651 - val_accuracy: 0.4000 - val_loss: 1.2024 - learning_rate: 3.0000e-04
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 

## 8) Evaluation (Accuracy + Macro-F1, Confusion Matrix)


In [11]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    _HAS_SEABORN = True
except Exception:
    _HAS_SEABORN = False

# CNN Evaluation
probs = cnn.predict(X_test_n, verbose=0)
y_pred_cnn = probs.argmax(axis=1)

cnn_acc = accuracy_score(y_test, y_pred_cnn)
cnn_f1_macro = f1_score(y_test, y_pred_cnn, average="macro")

print("CNN Accuracy:", cnn_acc)
print("CNN Macro-F1:", cnn_f1_macro)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_cnn))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_cnn,
    target_names=[inv_label_map[i] for i in sorted(inv_label_map)],
    zero_division=0,
))


CNN Accuracy: 0.43
CNN Macro-F1: 0.42001112501738286

Confusion Matrix:
[[24 31 11]
 [34 56 14]
 [17 64 49]]

Classification Report:
              precision    recall  f1-score   support

     pushups       0.32      0.36      0.34        66
      squats       0.37      0.54      0.44       104
      situps       0.66      0.38      0.48       130

    accuracy                           0.43       300
   macro avg       0.45      0.43      0.42       300
weighted avg       0.49      0.43      0.44       300



## 9) Robuste Evaluation: Mehrere zufällige Session-Splits (Accuracy, Macro-F1, 95%-CI)


In [12]:
from sklearn.utils.class_weight import compute_class_weight

def split_sessions_random(seed, test_size=TEST_SIZE, val_size_of_rest=VAL_SIZE_OF_REST):
    sessions_local = list_sessions(ROOT)

    outer_split = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=seed,
    )
    idx_train_val, idx_test = next(outer_split.split(sessions_local, groups=sessions_local))
    train_val = [sessions_local[i] for i in idx_train_val]
    test = [sessions_local[i] for i in idx_test]

    inner_split = GroupShuffleSplit(
        n_splits=1,
        test_size=val_size_of_rest,
        random_state=seed + 1,
    )
    idx_train, idx_val = next(inner_split.split(train_val, groups=train_val))
    train = [train_val[i] for i in idx_train]
    val = [train_val[i] for i in idx_val]

    return train, val, test


def make_ci95(series):
    n = len(series)
    if n < 2:
        return 0.0
    return 1.96 * float(np.std(series, ddof=1)) / math.sqrt(n)


def train_eval_cnn_once(X_train_n_i, y_train_i, X_val_n_i, y_val_i, X_test_n_i, y_test_i, seed, cfg):
    tf.keras.backend.clear_session()

    classes_i = np.unique(y_train_i)
    weights_i = compute_class_weight(
        class_weight="balanced",
        classes=classes_i,
        y=y_train_i,
    )
    class_weights_i = dict(zip(classes_i, weights_i))

    model = build_cnn(
        input_shape=(WIN, X_train_n_i.shape[-1]),
        num_classes=len(label_map),
        model_variant=cfg.get("model_variant", MODEL_VARIANT),
        base_filters=cfg.get("base_filters", BASE_FILTERS),
        dropout_head=cfg.get("dropout_head", DROPOUT_HEAD),
        dense_units=cfg.get("dense_units", DENSE_UNITS),
        learning_rate=cfg.get("learning_rate", LEARNING_RATE),
        weight_decay=cfg.get("weight_decay", WEIGHT_DECAY),
        use_focal_loss=cfg.get("use_focal_loss", USE_FOCAL_LOSS),
        focal_gamma=cfg.get("focal_gamma", FOCAL_GAMMA),
    )

    batch_size = cfg.get("batch_size", 32)
    epochs = cfg.get("epochs", 50)

    train_ds_i = tf.data.Dataset.from_tensor_slices((X_train_n_i, y_train_i))
    train_ds_i = train_ds_i.shuffle(min(8192, len(X_train_n_i)), seed=seed).batch(batch_size).map(augment).prefetch(tf.data.AUTOTUNE)
    val_ds_i = tf.data.Dataset.from_tensor_slices((X_val_n_i, y_val_i)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    ckpt = f"/tmp/cnn_seed_{seed}_{cfg.get('name', 'cfg')}.keras"
    callbacks_i = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=ckpt,
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=0,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-5,
        ),
    ]

    class_weights_for_fit = None if cfg.get("use_focal_loss", USE_FOCAL_LOSS) else class_weights_i

    history_i = model.fit(
        train_ds_i,
        validation_data=val_ds_i,
        epochs=epochs,
        class_weight=class_weights_for_fit,
        callbacks=callbacks_i,
        verbose=0,
    )

    y_pred_cnn_i = model.predict(X_test_n_i, verbose=0).argmax(axis=1)
    cnn_acc = accuracy_score(y_test_i, y_pred_cnn_i)
    cnn_f1 = f1_score(y_test_i, y_pred_cnn_i, average="macro")
    best_val_acc = float(np.max(history_i.history["val_accuracy"]))

    return cnn_acc, cnn_f1, best_val_acc


def run_one_split(seed, cfg=None):
    cfg = cfg or {}
    np.random.seed(seed)
    tf.random.set_seed(seed)

    train_sessions_i, val_sessions_i, test_sessions_i = split_sessions_random(seed)

    X_train_i, y_train_i = build_dataset(train_sessions_i, split_name=f"train-{seed}", sensors=SENSORS, verbose=False)
    X_val_i, y_val_i = build_dataset(val_sessions_i, split_name=f"val-{seed}", sensors=SENSORS, verbose=False)
    X_test_i, y_test_i = build_dataset(test_sessions_i, split_name=f"test-{seed}", sensors=SENSORS, verbose=False)

    mu = X_train_i.mean(axis=(0, 1), keepdims=True)
    sigma = X_train_i.std(axis=(0, 1), keepdims=True) + 1e-8

    X_train_n_i = (X_train_i - mu) / sigma
    X_val_n_i = (X_val_i - mu) / sigma
    X_test_n_i = (X_test_i - mu) / sigma

    # Random Forest
    rf_i = RandomForestClassifier(
        n_estimators=cfg.get("rf_n_estimators", 300),
        random_state=seed,
        n_jobs=-1,
    )
    rf_i.fit(X_train_n_i.reshape(X_train_n_i.shape[0], -1), y_train_i)
    y_pred_rf_i = rf_i.predict(X_test_n_i.reshape(X_test_n_i.shape[0], -1))
    rf_acc = accuracy_score(y_test_i, y_pred_rf_i)
    rf_f1 = f1_score(y_test_i, y_pred_rf_i, average="macro")

    # CNN
    cnn_acc, cnn_f1, best_val_acc = train_eval_cnn_once(
        X_train_n_i, y_train_i, X_val_n_i, y_val_i, X_test_n_i, y_test_i, seed, cfg
    )

    return {
        "seed": seed,
        "n_train_sessions": len(train_sessions_i),
        "n_val_sessions": len(val_sessions_i),
        "n_test_sessions": len(test_sessions_i),
        "rf_acc": rf_acc,
        "rf_f1_macro": rf_f1,
        "cnn_acc": cnn_acc,
        "cnn_f1_macro": cnn_f1,
        "cnn_best_val_acc": best_val_acc,
    }


SEEDS = [11, 22, 33, 42, 55, 66, 77, 88, 99, 111]
EVAL_CFG = {
    "name": "resnet_default",
    "model_variant": MODEL_VARIANT,
    "base_filters": BASE_FILTERS,
    "dropout_head": DROPOUT_HEAD,
    "dense_units": DENSE_UNITS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs": 50,
    "batch_size": 32,
    "use_focal_loss": USE_FOCAL_LOSS,
    "focal_gamma": FOCAL_GAMMA,
}

results = []
for s in SEEDS:
    row = run_one_split(seed=s, cfg=EVAL_CFG)
    results.append(row)
    print(
        f"Seed {s}: "
        f"RF acc={row['rf_acc']:.3f}, RF f1={row['rf_f1_macro']:.3f}, "
        f"CNN acc={row['cnn_acc']:.3f}, CNN f1={row['cnn_f1_macro']:.3f}, "
        f"best_val={row['cnn_best_val_acc']:.3f}"
    )

results_df = pd.DataFrame(results).sort_values("seed").reset_index(drop=True)
print("\nPer-Split Ergebnisse:")
print(results_df)

print("\nZusammenfassung (mean ± std, 95%-CI):")
for metric in ["rf_acc", "rf_f1_macro", "cnn_acc", "cnn_f1_macro", "cnn_best_val_acc"]:
    m = float(results_df[metric].mean())
    s = float(results_df[metric].std(ddof=1))
    ci = make_ci95(results_df[metric].values)
    print(f"{metric:16s}: {m:.3f} ± {s:.3f} (95%-CI ± {ci:.3f})")


Seed 11: RF acc=0.434, RF f1=0.386, CNN acc=0.514, CNN f1=0.485, best_val=0.540
Seed 22: RF acc=0.464, RF f1=0.348, CNN acc=0.457, CNN f1=0.379, best_val=0.498
Seed 33: RF acc=0.452, RF f1=0.397, CNN acc=0.476, CNN f1=0.444, best_val=0.535
Seed 42: RF acc=0.380, RF f1=0.317, CNN acc=0.417, CNN f1=0.376, best_val=0.472
Seed 55: RF acc=0.448, RF f1=0.342, CNN acc=0.525, CNN f1=0.438, best_val=0.593
Seed 66: RF acc=0.523, RF f1=0.445, CNN acc=0.520, CNN f1=0.508, best_val=0.576
Seed 77: RF acc=0.487, RF f1=0.450, CNN acc=0.380, CNN f1=0.378, best_val=0.570
Seed 88: RF acc=0.434, RF f1=0.398, CNN acc=0.459, CNN f1=0.348, best_val=0.512
Seed 99: RF acc=0.459, RF f1=0.423, CNN acc=0.459, CNN f1=0.340, best_val=0.453
Seed 111: RF acc=0.406, RF f1=0.349, CNN acc=0.413, CNN f1=0.325, best_val=0.638

Per-Split Ergebnisse:
   seed  n_train_sessions  n_val_sessions  n_test_sessions    rf_acc  \
0    11                12               4                5  0.434483   
1    22                12       

## 10) Hyperparameter-Optimierung (Random Search auf mehreren Seeds)


In [13]:
# Hinweis: Diese Suche ist rechenintensiv. Für schnellen Start mit kleinen Budgets laufen lassen.
SEARCH_SPACE = {
    "model_variant": ["resnet", "baseline"],
    "base_filters": [32, 48, 64],
    "dropout_head": [0.30, 0.40, 0.50],
    "dense_units": [64, 128, 192],
    "learning_rate": [1e-3, 5e-4, 3e-4],
    "weight_decay": [1e-6, 1e-5, 3e-5],
    "batch_size": [16, 32],
    "use_focal_loss": [False, True],
    "focal_gamma": [1.5, 2.0, 2.5],
}

HPO_TRIALS = 12
HPO_SEEDS = [11, 22, 33]
HPO_EPOCHS = 35

candidates = list(ParameterSampler(SEARCH_SPACE, n_iter=HPO_TRIALS, random_state=RANDOM_SEED))
hpo_rows = []

for idx, cfg in enumerate(candidates, start=1):
    cfg = dict(cfg)
    cfg["epochs"] = HPO_EPOCHS
    cfg["name"] = f"trial_{idx}"

    seed_rows = []
    print(f"\nTrial {idx}/{HPO_TRIALS}: {cfg}")
    for s in HPO_SEEDS:
        row = run_one_split(seed=s, cfg=cfg)
        seed_rows.append(row)

    df_trial = pd.DataFrame(seed_rows)
    trial_result = {
        "trial": idx,
        "cfg": cfg,
        "cnn_acc_mean": float(df_trial["cnn_acc"].mean()),
        "cnn_acc_std": float(df_trial["cnn_acc"].std(ddof=1)),
        "cnn_f1_mean": float(df_trial["cnn_f1_macro"].mean()),
        "cnn_val_mean": float(df_trial["cnn_best_val_acc"].mean()),
    }
    hpo_rows.append(trial_result)
    print(
        f" -> mean CNN acc={trial_result['cnn_acc_mean']:.3f}, "
        f"f1={trial_result['cnn_f1_mean']:.3f}, val={trial_result['cnn_val_mean']:.3f}"
    )

hpo_df = pd.DataFrame(hpo_rows).sort_values(["cnn_acc_mean", "cnn_f1_mean"], ascending=False).reset_index(drop=True)
print("\nTop-5 Trials:")
print(hpo_df.head(5)[["trial", "cnn_acc_mean", "cnn_acc_std", "cnn_f1_mean", "cnn_val_mean"]])

best_cfg = hpo_df.iloc[0]["cfg"]
print("\nBest Config:")
print(best_cfg)



Trial 1/12: {'weight_decay': 3e-05, 'use_focal_loss': False, 'model_variant': 'baseline', 'learning_rate': 0.0003, 'focal_gamma': 2.5, 'dropout_head': 0.4, 'dense_units': 192, 'batch_size': 16, 'base_filters': 32, 'epochs': 35, 'name': 'trial_1'}
 -> mean CNN acc=0.457, f1=0.397, val=0.498

Trial 2/12: {'weight_decay': 3e-05, 'use_focal_loss': False, 'model_variant': 'resnet', 'learning_rate': 0.0003, 'focal_gamma': 2.5, 'dropout_head': 0.4, 'dense_units': 128, 'batch_size': 32, 'base_filters': 64, 'epochs': 35, 'name': 'trial_2'}
 -> mean CNN acc=0.483, f1=0.409, val=0.535

Trial 3/12: {'weight_decay': 1e-06, 'use_focal_loss': False, 'model_variant': 'baseline', 'learning_rate': 0.001, 'focal_gamma': 2.0, 'dropout_head': 0.3, 'dense_units': 128, 'batch_size': 32, 'base_filters': 64, 'epochs': 35, 'name': 'trial_3'}
 -> mean CNN acc=0.510, f1=0.436, val=0.507

Trial 4/12: {'weight_decay': 1e-05, 'use_focal_loss': False, 'model_variant': 'baseline', 'learning_rate': 0.001, 'focal_gamma

## 11) Finale Bestätigung des besten HPO-Setups auf 10 Seeds


In [14]:
# Erwartet best_cfg aus der HPO-Zelle; fallback auf EVAL_CFG/default.
FINAL_SEEDS = [11, 22, 33, 42, 55, 66, 77, 88, 99, 111]

if "best_cfg" in globals():
    final_cfg = dict(best_cfg)
elif "EVAL_CFG" in globals():
    final_cfg = dict(EVAL_CFG)
else:
    final_cfg = {
        "model_variant": MODEL_VARIANT,
        "base_filters": BASE_FILTERS,
        "dropout_head": DROPOUT_HEAD,
        "dense_units": DENSE_UNITS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": 32,
        "use_focal_loss": USE_FOCAL_LOSS,
        "focal_gamma": FOCAL_GAMMA,
    }

final_cfg["epochs"] = 50
final_cfg["name"] = "best_final_10seeds"

print("Finales Setup:")
print("  SENSOR_MODE:", SENSOR_MODE)
print("  SENSORS:", SENSORS)
print("  WIN/STEP:", WIN, STEP)
print("  CFG:", final_cfg)

final_rows = []
for s in FINAL_SEEDS:
    row = run_one_split(seed=s, cfg=final_cfg)
    final_rows.append(row)
    print(f"Seed {s}: CNN acc={row['cnn_acc']:.3f}, CNN f1={row['cnn_f1_macro']:.3f}")

final_df = pd.DataFrame(final_rows)
print("\nFinale Kennzahlen (best_cfg, 10 Seeds):")
for metric in ["cnn_acc", "cnn_f1_macro", "cnn_best_val_acc"]:
    m = float(final_df[metric].mean())
    s = float(final_df[metric].std(ddof=1))
    ci = make_ci95(final_df[metric].values)
    print(f"{metric:16s}: {m:.3f} ± {s:.3f} (95%-CI ± {ci:.3f})")


Finales Setup:
  SENSOR_MODE: manual
  SENSORS: ['sw_r']
  WIN/STEP: 192 64
  CFG: {'weight_decay': 1e-05, 'use_focal_loss': True, 'model_variant': 'baseline', 'learning_rate': 0.0003, 'focal_gamma': 2.5, 'dropout_head': 0.4, 'dense_units': 128, 'batch_size': 16, 'base_filters': 64, 'epochs': 50, 'name': 'best_final_10seeds'}
Seed 11: CNN acc=0.438, CNN f1=0.373
Seed 22: CNN acc=0.497, CNN f1=0.356
Seed 33: CNN acc=0.521, CNN f1=0.412
Seed 42: CNN acc=0.467, CNN f1=0.336
Seed 55: CNN acc=0.470, CNN f1=0.380
Seed 66: CNN acc=0.573, CNN f1=0.518
Seed 77: CNN acc=0.462, CNN f1=0.427
Seed 88: CNN acc=0.478, CNN f1=0.428
Seed 99: CNN acc=0.449, CNN f1=0.354
Seed 111: CNN acc=0.428, CNN f1=0.374

Finale Kennzahlen (best_cfg, 10 Seeds):
cnn_acc         : 0.478 ± 0.043 (95%-CI ± 0.027)
cnn_f1_macro    : 0.396 ± 0.053 (95%-CI ± 0.033)
cnn_best_val_acc: 0.549 ± 0.050 (95%-CI ± 0.031)


## 12) A/B-Test: `sw_r` vs `auto_common` (identische Seeds, gleiches Modell)


In [15]:
# Nutzt best_cfg aus Abschnitt 10/11 falls vorhanden, sonst EVAL_CFG/default.
if "best_cfg" in globals():
    AB_CFG = dict(best_cfg)
elif "EVAL_CFG" in globals():
    AB_CFG = dict(EVAL_CFG)
else:
    AB_CFG = {
        "model_variant": MODEL_VARIANT,
        "base_filters": BASE_FILTERS,
        "dropout_head": DROPOUT_HEAD,
        "dense_units": DENSE_UNITS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": 32,
        "use_focal_loss": USE_FOCAL_LOSS,
        "focal_gamma": FOCAL_GAMMA,
    }

AB_CFG["epochs"] = 50
AB_SEEDS = FINAL_SEEDS if "FINAL_SEEDS" in globals() else (SEEDS if "SEEDS" in globals() else [11,22,33,42,55,66,77,88,99,111])


def choose_sensors(sensor_mode):
    if sensor_mode == "sw_r":
        return ["sw_r"]
    if sensor_mode == "auto_common":
        common = rank_sensors(get_common_sensors(ROOT), priority=SENSOR_PRIORITY)
        chosen = common[:SENSOR_MAX_COUNT] if SENSOR_MAX_COUNT is not None else common
        return chosen or ["sw_r"]
    raise ValueError(f"Unknown sensor_mode: {sensor_mode}")


def build_dataset_custom(session_list, sensors, win, step, split_name="split", verbose=False):
    X_all, y_all = [], []
    for wdir in session_list:
        features, labels = load_one_session(wdir, sensors=sensors)
        X, y = make_windows(features, labels, win=win, step=step)
        if len(X) == 0:
            continue
        X_all.append(X)
        y_all.append(y)
        if verbose:
            print(f"[{split_name}]", wdir, X.shape)

    if not X_all:
        raise ValueError(f"No windows for split {split_name}, sensors={sensors}, win={win}, step={step}")

    return np.concatenate(X_all), np.concatenate(y_all)


def train_eval_cnn_once_custom(X_train_n_i, y_train_i, X_val_n_i, y_val_i, X_test_n_i, y_test_i, seed, cfg, win):
    tf.keras.backend.clear_session()

    classes_i = np.unique(y_train_i)
    weights_i = compute_class_weight(class_weight="balanced", classes=classes_i, y=y_train_i)
    class_weights_i = dict(zip(classes_i, weights_i))

    model = build_cnn(
        input_shape=(win, X_train_n_i.shape[-1]),
        num_classes=len(label_map),
        model_variant=cfg.get("model_variant", MODEL_VARIANT),
        base_filters=cfg.get("base_filters", BASE_FILTERS),
        dropout_head=cfg.get("dropout_head", DROPOUT_HEAD),
        dense_units=cfg.get("dense_units", DENSE_UNITS),
        learning_rate=cfg.get("learning_rate", LEARNING_RATE),
        weight_decay=cfg.get("weight_decay", WEIGHT_DECAY),
        use_focal_loss=cfg.get("use_focal_loss", USE_FOCAL_LOSS),
        focal_gamma=cfg.get("focal_gamma", FOCAL_GAMMA),
    )

    batch_size = cfg.get("batch_size", 32)
    epochs = cfg.get("epochs", 50)

    train_ds_i = tf.data.Dataset.from_tensor_slices((X_train_n_i, y_train_i))
    train_ds_i = train_ds_i.shuffle(min(8192, len(X_train_n_i)), seed=seed).batch(batch_size).map(augment).prefetch(tf.data.AUTOTUNE)
    val_ds_i = tf.data.Dataset.from_tensor_slices((X_val_n_i, y_val_i)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    callbacks_i = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
    ]

    class_weights_for_fit = None if cfg.get("use_focal_loss", USE_FOCAL_LOSS) else class_weights_i

    history_i = model.fit(
        train_ds_i,
        validation_data=val_ds_i,
        epochs=epochs,
        class_weight=class_weights_for_fit,
        callbacks=callbacks_i,
        verbose=0,
    )

    y_pred = model.predict(X_test_n_i, verbose=0).argmax(axis=1)
    return (
        accuracy_score(y_test_i, y_pred),
        f1_score(y_test_i, y_pred, average="macro"),
        float(np.max(history_i.history["val_accuracy"])),
    )


def run_one_split_custom(seed, cfg, sensors, win, step):
    np.random.seed(seed)
    tf.random.set_seed(seed)

    train_sessions_i, val_sessions_i, test_sessions_i = split_sessions_random(seed)

    X_train_i, y_train_i = build_dataset_custom(train_sessions_i, sensors=sensors, win=win, step=step)
    X_val_i, y_val_i = build_dataset_custom(val_sessions_i, sensors=sensors, win=win, step=step)
    X_test_i, y_test_i = build_dataset_custom(test_sessions_i, sensors=sensors, win=win, step=step)

    mu = X_train_i.mean(axis=(0, 1), keepdims=True)
    sigma = X_train_i.std(axis=(0, 1), keepdims=True) + 1e-8
    X_train_n_i = (X_train_i - mu) / sigma
    X_val_n_i = (X_val_i - mu) / sigma
    X_test_n_i = (X_test_i - mu) / sigma

    cnn_acc, cnn_f1, best_val = train_eval_cnn_once_custom(
        X_train_n_i, y_train_i, X_val_n_i, y_val_i, X_test_n_i, y_test_i, seed, cfg, win
    )

    return {
        "seed": seed,
        "cnn_acc": cnn_acc,
        "cnn_f1_macro": cnn_f1,
        "cnn_best_val_acc": best_val,
        "n_train": len(X_train_i),
        "n_val": len(X_val_i),
        "n_test": len(X_test_i),
    }


def summarize_df(df, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols):
        keys = (keys,) if not isinstance(keys, tuple) else keys
        d = {col: val for col, val in zip(group_cols, keys)}
        for m in ["cnn_acc", "cnn_f1_macro", "cnn_best_val_acc"]:
            d[f"{m}_mean"] = float(g[m].mean())
            d[f"{m}_std"] = float(g[m].std(ddof=1))
            d[f"{m}_ci95"] = make_ci95(g[m].values)
        rows.append(d)
    return pd.DataFrame(rows)

ab_runs = []
for mode in ["sw_r", "auto_common"]:
    sensors_mode = choose_sensors(mode)
    print(f"\nA/B mode={mode}, sensors={sensors_mode}")
    for s in AB_SEEDS:
        row = run_one_split_custom(seed=s, cfg=AB_CFG, sensors=sensors_mode, win=WIN, step=STEP)
        row["sensor_mode"] = mode
        row["sensors"] = ",".join(sensors_mode)
        row["win"] = WIN
        row["step"] = STEP
        ab_runs.append(row)
        print(f"  seed {s}: acc={row['cnn_acc']:.3f}, f1={row['cnn_f1_macro']:.3f}")

ab_df = pd.DataFrame(ab_runs)
ab_summary = summarize_df(ab_df, ["sensor_mode", "sensors", "win", "step"]).sort_values("cnn_f1_macro_mean", ascending=False)

print("\nA/B Zusammenfassung (nach Macro-F1 sortiert):")
print(ab_summary[["sensor_mode", "sensors", "cnn_acc_mean", "cnn_acc_std", "cnn_f1_macro_mean", "cnn_f1_macro_std", "cnn_f1_macro_ci95"]])

BEST_SENSOR_MODE = ab_summary.iloc[0]["sensor_mode"]
BEST_SENSORS = ab_summary.iloc[0]["sensors"].split(",")
print("\nBestes Sensor-Setup:", BEST_SENSOR_MODE, BEST_SENSORS)



A/B mode=sw_r, sensors=['sw_r']
  seed 11: acc=0.421, f1=0.328
  seed 22: acc=0.510, f1=0.366
  seed 33: acc=0.551, f1=0.498
  seed 42: acc=0.477, f1=0.400
  seed 55: acc=0.503, f1=0.402
  seed 66: acc=0.573, f1=0.525
  seed 77: acc=0.477, f1=0.456
  seed 88: acc=0.512, f1=0.442
  seed 99: acc=0.414, f1=0.413
  seed 111: acc=0.449, f1=0.337

A/B mode=auto_common, sensors=['sw_r', 'sw_l', 'eb_l']
  seed 11: acc=0.497, f1=0.377
  seed 22: acc=0.437, f1=0.388
  seed 33: acc=0.446, f1=0.428
  seed 42: acc=0.513, f1=0.490
  seed 55: acc=0.413, f1=0.399
  seed 66: acc=0.536, f1=0.512
  seed 77: acc=0.394, f1=0.377
  seed 88: acc=0.556, f1=0.418
  seed 99: acc=0.494, f1=0.454
  seed 111: acc=0.389, f1=0.329

A/B Zusammenfassung (nach Macro-F1 sortiert):
   sensor_mode         sensors  cnn_acc_mean  cnn_acc_std  cnn_f1_macro_mean  \
0  auto_common  sw_r,sw_l,eb_l      0.467523     0.059828           0.417251   
1         sw_r            sw_r      0.488544     0.051869           0.416635   

 

## 13) Windowing-Grid mit bestem Sensor-Setup


In [16]:
WINDOW_GRID = [
    (128, 64),
    (192, 64),
    (256, 64),
]

window_runs = []
for win_i, step_i in WINDOW_GRID:
    print(f"\nWindowing: win={win_i}, step={step_i}, sensors={BEST_SENSORS}")
    for s in AB_SEEDS:
        row = run_one_split_custom(seed=s, cfg=AB_CFG, sensors=BEST_SENSORS, win=win_i, step=step_i)
        row["sensor_mode"] = BEST_SENSOR_MODE
        row["sensors"] = ",".join(BEST_SENSORS)
        row["win"] = win_i
        row["step"] = step_i
        window_runs.append(row)
        print(f"  seed {s}: acc={row['cnn_acc']:.3f}, f1={row['cnn_f1_macro']:.3f}")

window_df = pd.DataFrame(window_runs)
window_summary = summarize_df(window_df, ["win", "step", "sensor_mode", "sensors"]).sort_values("cnn_f1_macro_mean", ascending=False)

print("\nWindowing-Zusammenfassung (nach Macro-F1 sortiert):")
print(window_summary[["win", "step", "cnn_acc_mean", "cnn_acc_std", "cnn_f1_macro_mean", "cnn_f1_macro_std", "cnn_f1_macro_ci95"]])

BEST_WINDOW = (int(window_summary.iloc[0]["win"]), int(window_summary.iloc[0]["step"]))
print("\nBestes Windowing:", BEST_WINDOW)



Windowing: win=128, step=64, sensors=['sw_r', 'sw_l', 'eb_l']
  seed 11: acc=0.397, f1=0.345
  seed 22: acc=0.438, f1=0.410
  seed 33: acc=0.424, f1=0.349
  seed 42: acc=0.523, f1=0.498
  seed 55: acc=0.443, f1=0.413
  seed 66: acc=0.556, f1=0.507
  seed 77: acc=0.368, f1=0.312
  seed 88: acc=0.416, f1=0.296
  seed 99: acc=0.448, f1=0.418
  seed 111: acc=0.405, f1=0.390

Windowing: win=192, step=64, sensors=['sw_r', 'sw_l', 'eb_l']
  seed 11: acc=0.486, f1=0.386
  seed 22: acc=0.487, f1=0.454
  seed 33: acc=0.455, f1=0.433
  seed 42: acc=0.357, f1=0.353
  seed 55: acc=0.434, f1=0.371
  seed 66: acc=0.589, f1=0.559
  seed 77: acc=0.455, f1=0.419
  seed 88: acc=0.434, f1=0.331
  seed 99: acc=0.471, f1=0.438
  seed 111: acc=0.551, f1=0.459

Windowing: win=256, step=64, sensors=['sw_r', 'sw_l', 'eb_l']
  seed 11: acc=0.396, f1=0.392
  seed 22: acc=0.471, f1=0.408
  seed 33: acc=0.443, f1=0.348
  seed 42: acc=0.557, f1=0.507
  seed 55: acc=0.502, f1=0.391
  seed 66: acc=0.561, f1=0.494
  s